# PLM4NewsRS on Colab: every BERT model on every dataset

For each **dataset × model** pair, this notebook:

1. trains the clean model (3 seeds) and one model per poisoned set (`--train-all`),
2. evaluates them on every stage6 benchmark (`--generate-report`),
3. copies **results, reports, training dynamics, TensorBoard logs and each run's final `-last.ckpt`** to Google Drive,
4. writes a "done" marker on Drive and frees the local disk.

Training runs on Colab's local disk, because the per-epoch checkpoints (~440 MB each, used by the evaluation) don't fit on Drive. Only the final checkpoint of each run goes to Drive, about **40 GB per dataset**, so make sure your Drive has room.

**Before you start**
- Runtime → Change runtime type → **GPU**.
- Zip your fakeun-dataprep output on your computer, e.g. `cd fakeun-dataprep/output && zip -rq problem2.zip problem2`. Then drag `problem2.zip` into the Files pane (left sidebar, `/content`), or put it on Drive and set `DATA_ZIP` below. Any folder containing a `news_items.csv` counts as a dataset.

**After a disconnect:** reconnect and choose *Runtime → Run all* again. Finished pairs are skipped (from their markers on Drive). A pair that was cut off is trained again from the start, because its epoch checkpoints were on the lost local disk.

In [ ]:
# ===================== Settings =====================
REPO_URL  = "https://github.com/thamolwanpo/plm4newsrs.git"
BRANCH    = "feature/skip-existing-runs"   # change to the branch it is merged into, once merged
DRIVE_OUT = "/content/drive/MyDrive/plm4newsrs_outputs"  # results, logs, final checkpoints

DATA_ZIP  = ""      # "" = use the .zip files you uploaded to /content; or a zip on Drive, e.g. "/content/drive/MyDrive/problem2.zip"
DATASETS  = None    # None = every dataset found in the data; or a list, e.g. ["politifact", "gossipcop"]

# Model name -> template config. The config's dataset is overridden by the loop below,
# so one config per model is enough (the *_politifact / *_gossipcop pairs are otherwise identical).
MODELS = {
    "simple_bert_frozen":   "configs/experiments/simple/bert_frozen.yaml",
    "simple_bert_finetune": "configs/experiments/simple/bert_finetune.yaml",
    "naml_bert_frozen":     "configs/experiments/naml/naml_bert_frozen_politifact.yaml",
    "naml_bert_finetune":   "configs/experiments/naml/naml_bert_finetune_politifact.yaml",
    "nrms_bert_frozen":     "configs/experiments/nrms/nrms_bert_frozen_politifact.yaml",
    "nrms_bert_finetune":   "configs/experiments/nrms/nrms_bert_finetune_politifact.yaml",
    "lstur_bert_frozen":    "configs/experiments/lstur/lstur_bert_frozen_politifact.yaml",
    "lstur_bert_finetune":  "configs/experiments/lstur/lstur_bert_finetune_politifact.yaml",
}

KEEP_EPOCH_CHECKPOINTS_ON_DRIVE = False  # True also copies every clean-model epoch checkpoint (~2.5x the space)

## 1. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ No GPU: Runtime → Change runtime type → GPU"
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "{DRIVE_OUT}"
!df -h /content | tail -1

## 2. Code and packages
If the repository is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) and turn on notebook access for it.

In [ ]:
import os, sys

REPO = "/content/plm4newsrs"
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass  # no token: public repository

if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q --branch {BRANCH} {url} {REPO}
!git -C {REPO} log --oneline -1

# Colab already has torch, transformers, pandas, scikit-learn, matplotlib, seaborn, pyarrow, tensorboard
!pip install -q lightning torchmetrics

%cd {REPO}
if REPO not in sys.path:
    sys.path.insert(0, REPO)

## 3. Data

In [ ]:
import glob, shutil, zipfile
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def find_datasets():
    return {p.parent.name: p.parent for p in sorted(DATA_DIR.rglob("news_items.csv"))
            if "__MACOSX" not in p.parts}

if not find_datasets():
    zips = [DATA_ZIP] if DATA_ZIP else sorted(glob.glob("/content/*.zip"))
    if not zips:
        raise FileNotFoundError("No data: upload your zip to /content (Files pane) or set DATA_ZIP")
    for z in zips:
        if z.startswith("/content/drive/"):  # copying first is much faster than reading from Drive
            z = shutil.copy(z, "/content/")
        print("Unzipping", z)
        zipfile.ZipFile(z).extractall(DATA_DIR)

data_roots = find_datasets()
datasets = DATASETS or sorted(data_roots)
missing = [d for d in datasets if d not in data_roots]
assert not missing, f"Datasets not found in the data: {missing} (found: {sorted(data_roots)})"

for d in datasets:
    root = data_roots[d]
    n_poison = len([p for p in (root / "stage4").glob("A_*") if (p / "train_model.csv").exists()])
    n_bench = len(list((root / "stage6").glob("*.csv")))
    print(f"{d:15s} {root}   poisoned sets: {n_poison}   stage6 benchmarks: {n_bench}")

## 4. Plan: what is already done

In [ ]:
import pandas as pd
from configs import load_config
from src.training.trainer import get_training_runs, is_run_complete

LOCAL_OUT = Path("/content/outputs")

def make_config(model, dataset):
    cfg = load_config(MODELS[model])
    cfg.dataset = dataset
    cfg.data_root = data_roots[dataset]
    cfg.base_dir = LOCAL_OUT / dataset
    cfg.__post_init__()
    return cfg

def run_names(cfg):
    return [name or f"poisoned_{poison_set}" for _, poison_set, _, name in get_training_runs(cfg)]

def done_marker(model, dataset):
    return Path(DRIVE_OUT) / dataset / "_done" / f"{model}.json"

def status_table():
    rows = []
    for d in datasets:
        for m in MODELS:
            cfg = make_config(m, d)
            rows.append({"dataset": d, "model": m, "runs": len(run_names(cfg)),
                         "output folder": f"{cfg.experiment_name}/{cfg.get_lm_folder_name()}",
                         "done": "✅" if done_marker(m, d).exists() else ""})
    return pd.DataFrame(rows)

plan = status_table()
print(f"{(plan.done == '').sum()} of {len(plan)} dataset × model pairs to do, "
      f"{plan[plan.done == ''].runs.sum()} training runs")
plan

## 5. Train, evaluate and save everything
This cell runs for many hours and will likely need more than one Colab session (12 h free, 24 h Pro). Only key lines are printed. The full output of each step goes to `DRIVE_OUT/_logs/{dataset}/`.

In [ ]:
import datetime, json, re, subprocess, time

SHOW = re.compile(r"TRAINING \[|✅|❌|⚠️|⏭️|Metric val_auc|Traceback|Error|Evaluating|Results saved")

def run(cmd, log_path):
    """Run cmd, write its full output to log_path (on Drive), print only key lines."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = []
    with open(log_path, "ab") as log:
        log.write(f"\n===== {datetime.datetime.now():%Y-%m-%d %H:%M:%S}  {' '.join(cmd)}\n".encode())
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                env=dict(os.environ, PYTHONUNBUFFERED="1"))
        for raw in proc.stdout:
            # progress bars redraw with \r: keep only their final state
            line = raw.rstrip(b"\r\n").split(b"\r")[-1].decode("utf-8", "replace").rstrip()
            if not line:
                continue
            log.write(line.encode() + b"\n")
            tail = (tail + [line])[-40:]
            if SHOW.search(line):
                print("    " + line[:200], flush=True)
                log.flush()
        rc = proc.wait()
    if rc != 0:
        print(f"    ❌ exit code {rc}. Last lines of {log_path}:\n" + "\n".join(tail))
    return rc

ignore = None if KEEP_EPOCH_CHECKPOINTS_ON_DRIVE else shutil.ignore_patterns("*-epoch=*.ckpt")

for dataset in datasets:
    for model, config_path in MODELS.items():
        marker = done_marker(model, dataset)
        if marker.exists():
            print(f"⏭️  {dataset} / {model}: already done")
            continue

        cfg = make_config(model, dataset)
        rel = Path(cfg.experiment_name) / cfg.get_lm_folder_name()
        local_dir, drive_dir = cfg.base_dir / rel, Path(DRIVE_OUT) / dataset / rel
        logs = Path(DRIVE_OUT) / "_logs" / dataset
        common = ["--config", config_path, "--dataset", dataset, "--data-root", str(cfg.data_root)]
        start = time.time()
        print(f"\n{'=' * 80}\n▶ {dataset} / {model}   ({datetime.datetime.now():%H:%M})\n{'=' * 80}")

        # Train (finished runs on the local disk are skipped, e.g. when this cell is rerun)
        run([sys.executable, "scripts/train.py", *common, "--train-all", "--skip-existing",
             "--base-dir", str(cfg.base_dir)], logs / f"{model}_train.txt")
        unfinished = [n for n in run_names(cfg) if not is_run_complete(cfg.get_paths(), n)]

        # Evaluate every checkpoint (clean per-epoch ones included) on every benchmark
        rc_eval = run([sys.executable, "scripts/evaluate.py", *common, "--generate-report",
                       "--output-dir", str(cfg.base_dir)], logs / f"{model}_eval.txt")

        # Copy results, logs and final checkpoints to Drive
        print(f"    Copying to {drive_dir} ...", flush=True)
        shutil.copytree(local_dir, drive_dir, ignore=ignore, dirs_exist_ok=True)
        minutes = (time.time() - start) / 60

        if unfinished or rc_eval != 0:
            print(f"    ⚠️ Not marked done ({minutes:.0f} min). Unfinished runs: {unfinished or 'none'}; "
                  f"evaluation exit code {rc_eval}. Check the logs, then rerun this cell: "
                  "finished runs are skipped.")
            continue
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps({"finished": datetime.datetime.now().isoformat(),
                                      "minutes": round(minutes), "runs": run_names(cfg)}, indent=1))
        shutil.rmtree(local_dir)  # free the local disk for the next pair
        print(f"    ✅ Done in {minutes:.0f} min")

print("\nAll done." if all(done_marker(m, d).exists() for d in datasets for m in MODELS)
      else "\nSome pairs are not done: see the ⚠️ lines above.")

## 6. Status

In [ ]:
status_table()

## 7. Download
Downloading a folder of many files from Drive is slow, so this cell writes one zip **without checkpoints** next to `DRIVE_OUT`. Download it from drive.google.com. For the checkpoints, use Google Drive for desktop, which syncs the folder to your computer.

In [ ]:
out = Path(DRIVE_OUT)
zip_path = out.parent / f"{out.name}_results.zip"
!cd "{out.parent}" && rm -f "{zip_path}" && zip -rq "{zip_path}" "{out.name}" -x "*.ckpt"
!ls -lh "{zip_path}"
# Small zips can also be downloaded straight from here:
# from google.colab import files; files.download(str(zip_path))

## Optional: TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{DRIVE_OUT}"